# `c11_hr` — Human Resources, Staffing Mix, and Tenure

**Component curation notebook.** Fetches the raw IPEDS distribution files, verifies the
reference period against official documentation, locks the schema, reshapes to the
declared grain, validates, and writes one curated table with a metadata sidecar.

| Property | Value |
|---|---|
| Native tables | `S2023_OC`, `S2023_IS`, `S2023_SIS`, `S2023_NH`, `SAL2023_IS`, `DRVHR2023` |
| Reference period | Payroll snapshot of November 1, 2023; salary outlays for AY 2023-24 |
| Curated grain | `UNITID` x `STAFFCAT` |
| Output | `data/curated/c11_hr.parquet` |

Use S2023_OC for the standing stock of employees by occupational category and S2023_SIS for tenure status by FACSTAT. These are snapshots of who is employed. Note that STAFFCAT bundles occupation with full- and part-time status, while OCCUPCAT and FTPT carry those two dimensions separately; prefer the separated columns when building features, and use SABDTYPE only when you need continuity with the pre-2012 occupational coding.

> **Pitfall.** S2023_NH is a flow, not a stock: it counts people hired between November 1, 2022 and October 31, 2023. Using it for staffing mix or tenure density measures the composition of one year's hiring rather than the composition of the workforce, and the two differ most at exactly the institutions undergoing change. Note also that STAFFCAT includes nested aggregate codes; summing across all of them double-counts.

## 1. Environment

One import surface, so a parsing quirk is fixed once rather than twelve times.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import ipeds_utils as iu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

SLUG = "c11_hr"
TABLES = ['S2023_OC', 'S2023_IS', 'S2023_SIS', 'S2023_NH', 'SAL2023_IS', 'DRVHR2023']
GRAIN = ['UNITID', 'STAFFCAT']
REFERENCE_PERIOD = 'Payroll snapshot of November 1, 2023; salary outlays for AY 2023-24'

print("ipeds_utils", iu.__version__, "| pandas", pd.__version__)

ipeds_utils 1.1.0 | pandas 3.0.5


## 2. Retrieve

Downloads are cached, so re-running this notebook is offline and cheap. Every retrieval returns a provenance record carrying a SHA-256 digest, which is what makes a result reproducible rather than merely repeatable.

In [2]:
RAW_DIR = "../data/raw"   # relative to notebooks/, so all twelve share one cache

provenance = [iu.fetch(t, raw_dir=RAW_DIR) for t in TABLES]
pd.DataFrame(provenance)[["table", "data_bytes", "data_sha256", "retrieved_utc"]]

,table,data_bytes,data_sha256,retrieved_utc
0,S2023_OC,2625688,8a18c9092cd8a966e9fc50c36b98265bfe06072292ff8e...,2026-09-24T17:19:35+00:00
1,S2023_IS,899888,ad32dfa87dd498dc4b6d0cd6bd64059473a43c87dcc7b0...,2026-09-24T17:19:35+00:00
2,S2023_SIS,123621,72db9cbbeecd9558bc2d52af90194693f89fef9467fac6...,2026-09-24T17:19:35+00:00
3,S2023_NH,513466,8411cc2d8616d4cf559cda4b192ecec97ed7ea0905700d...,2026-09-24T17:19:35+00:00
4,SAL2023_IS,987323,39336a8fd8004993e5e87e9585dec422f7a55101d165fb...,2026-09-24T17:19:35+00:00
5,DRVHR2023,151227,66a912023cd57de0ed4726e087efb1c5bb5bb23cb66ddd...,2026-09-24T17:19:35+00:00


## 3. Verify the reference period

**Do not skip this cell.** The filename year is not the reference period, and the offsets are not uniform across components. This assertion fails loudly rather than letting a misaligned period corrupt every downstream year comparison, where it would be invisible in the data itself.

In [3]:
intro = iu.assert_reference_period(
    provenance[0]["dict_path"],
    expect=r'(Fall 2023|November 1)',
    table=TABLES[0],
)
print(intro[:600])

File documentation for full- and part-time staff by occupational category, race/ethnicity, and gender: Fall 2023
(Provisional release)
Filename S2023_OC
Overview This file contains the number of staff on the payroll of the institution as of November 1,  by occupational categories, race/ethnicity and gender. This file has multiple records per institution.  Each record is uniquely defined by the variables IPEDS ID (UNITID), and occupational category and full- and part-time status (STAFFCAT).
Note Prior to the IPEDS Web-based data collection in 2001, human resource data reported by race/ethnicity


## 4. Inspect the dictionary

Variable labels come from the published dictionary, never from memory. This is also where value sets are read, so categorical decoding is driven by the official codebook and a taxonomy revision surfaces as unmatched codes instead of a plausible-looking wrong label.

In [4]:
variables = iu.read_dict(provenance[0]["dict_path"])
valuesets = iu.read_valuesets(provenance[0]["dict_path"])

print(f"{len(variables)} variables documented, {len(valuesets)} value-set rows")
variables[["varname", "vartitle"]].head(20)

35 variables documented, 104 value-set rows


,varname,vartitle
0,UNITID,Unique identification number of the institution
1,STAFFCAT,Occupation and full- and part-time status
2,FTPT,Full-time or part-time status
3,OCCUPCAT,Occupation category
4,SABDTYPE,Identifies occupations consistent with previou...
5,HRTOTLT,Grand total
6,HRTOTLM,Grand total men
7,HRTOTLW,Grand total women
8,HRAIANT,American Indian or Alaska Native total
9,HRAIANM,American Indian or Alaska Native men


## 5. Load and lock the schema

The first run records the column signature; later runs fail if it drifts.

In [5]:
KEEP = ['UNITID', 'STAFFCAT', 'FTPT', 'OCCUPCAT', 'SABDTYPE', 'HRTOTLT', 'HRTOTLM', 'HRTOTLW']

raw = iu.read_csv(provenance[0]["data_path"])
print("raw shape", raw.shape)

lock = iu.lock_schema(raw, TABLES[0], schema_dir="../schemas", strict=False)
print("schema:", lock["status"], "| added", lock["added"][:5], "| removed", lock["removed"][:5])

available = [c for c in KEEP if c in raw.columns]
missing = [c for c in KEEP if c not in raw.columns]
if missing:
    print("NOT PRESENT in this cycle (verify against the varlist above):", missing)

frame = raw[available].copy()
frame.head()

raw shape (180266, 65)
schema: unchanged | added [] | removed []


,UNITID,STAFFCAT,FTPT,OCCUPCAT,SABDTYPE,HRTOTLT,HRTOTLM,HRTOTLW
0,100654,1100,1,100,-2,1008,432,576
1,100654,1200,1,200,-2,384,190,194
2,100654,1210,1,210,-2,379,188,191
3,100654,1220,1,220,-2,5,2,3
4,100654,1250,1,250,-2,25,4,21


## 6. Mask reserved missing codes

IPEDS encodes missingness as negative integers. A mean computed without masking them is badly wrong and looks entirely plausible, which is what makes this the most costly single omission in IPEDS analysis.

In [6]:
RESERVED = [-1, -2, -3, -9]

numeric_cols = [
    c for c in frame.columns
    if c not in ("UNITID", *GRAIN) and pd.api.types.is_numeric_dtype(frame[c])
]

before = frame[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    frame.loc[frame[col].isin(RESERVED), col] = np.nan
after = frame[numeric_cols].isna().sum().sum()

# Masking turns an integer column into float (1 becomes 1.0). Measures can stay float,
# since NaN is what the models expect, but category codes go back to nullable integers
# so they print, join, and decode as codes rather than as 1.0.
for col in ['STAFFCAT', 'FTPT', 'OCCUPCAT']:
    if col in frame.columns and pd.api.types.is_float_dtype(frame[col]):
        if (frame[col].dropna() % 1 == 0).all():
            frame[col] = frame[col].astype("Int64")

print(f"masked {after - before:,} reserved-code cells across {len(numeric_cols)} numeric columns")

masked 163,587 reserved-code cells across 6 numeric columns


## 7. Carry the imputation flags

An imputed value and a reported value are not the same evidence. A column where most institutions carry a generated flag should not be modelled as though it were observed, and this is where that judgement becomes possible.

In [7]:
values, flags = iu.split_imputation_flags(raw, numeric_cols)

if flags.shape[1] > 1:
    summary = iu.imputation_summary(flags)
    display(summary.head(15))
    reported = summary[summary.flag == "R"].set_index("column")["share"]
    weak = reported[reported < 0.90]
    if len(weak):
        print("Columns under 90% reported — interpret with care:")
        display(weak)
else:
    print("No X-prefixed imputation flags accompany this file.")

,column,flag,n,share
3,XHRTOTLM,R,158865,0.8813
4,XHRTOTLM,Z,21235,0.1178
5,XHRTOTLM,P,118,0.0007
6,XHRTOTLM,N,48,0.0003
0,XHRTOTLT,R,180100,0.9991
1,XHRTOTLT,P,118,0.0007
2,XHRTOTLT,N,48,0.0003
7,XHRTOTLW,R,168993,0.9375
8,XHRTOTLW,Z,11107,0.0616
9,XHRTOTLW,P,118,0.0007


Columns under 90% reported — interpret with care:


column
XHRTOTLM    0.8813
Name: share, dtype: float64

## 8. Decode categoricals

Labels from the published value sets, not hand-typed mappings.

In [8]:
CATEGORICALS = ['STAFFCAT', 'FTPT', 'OCCUPCAT']

unresolved = {}
for col in CATEGORICALS:
    if col in frame.columns:
        frame = iu.decode(frame, valuesets, col)
        unmatched = frame.loc[frame[col].notna() & frame[f"{col}_LABEL"].isna(), col].unique()
        if len(unmatched):
            unresolved[col] = sorted(unmatched.tolist())[:10]

# An unmatched code means a taxonomy change or a parsing fault. Either way the
# labels are wrong, so this stops the notebook rather than printing a warning.
assert not unresolved, f"codes absent from the published value set: {unresolved}"

label_cols = [c for c in frame.columns if c.endswith("_LABEL")]
frame[CATEGORICALS + label_cols].drop_duplicates().head(20) if label_cols else frame.head()

,STAFFCAT,FTPT,OCCUPCAT,STAFFCAT_LABEL,FTPT_LABEL,OCCUPCAT_LABEL
0,1100,1,100,Grand Total,All staff,All staff
1,1200,1,200,"Instructional, research and public service",All staff,"Instructional, research and public service staff"
2,1210,1,210,Instructional staff,All staff,"Instructional staff, total"
3,1220,1,220,Research,All staff,Research
4,1250,1,250,"Librarians and Library Technicians/Archivists,...",All staff,Librarians/Library Technicians/Archivists and ...
5,1260,1,260,"Librarians and Library Technicians/Archivists,...",All staff,"Librarians, Curators, and Archivists"
6,1262,1,262,Librarians,All staff,Librarians
7,1263,1,263,Library Technicians,All staff,Library Technicians
8,1270,1,270,Student and Academic Affairs and Other Educati...,All staff,Student and Academic Affairs and Other Educati...
9,1300,1,300,Management,All staff,Management


## 9. Reshape to the declared grain

Target grain: `UNITID` x `STAFFCAT`. The grain is asserted, not assumed, because a duplicated key silently inflates every aggregate computed downstream.

In [9]:
curated = frame.copy()

# This component already arrives at its declared grain, so curation is a
# pass-through. Components with a long layout (GRTYPE, EFFYALEV, STAFFCAT,
# OMCHRT) filter or pivot here instead; see c10_f for a full worked reshape.

present_grain = [g for g in GRAIN if g in curated.columns]
duplicated = curated.duplicated(subset=present_grain, keep=False).sum()
print(f"grain {present_grain} -> {len(curated):,} rows, {duplicated} duplicated")
assert duplicated == 0, "Declared grain is not unique; resolve before continuing."

curated.head()

grain ['UNITID', 'STAFFCAT'] -> 180,266 rows, 0 duplicated


,UNITID,STAFFCAT,FTPT,OCCUPCAT,SABDTYPE,HRTOTLT,HRTOTLM,HRTOTLW,STAFFCAT_LABEL,FTPT_LABEL,OCCUPCAT_LABEL
0,100654,1100,1,100,NaN,1008.0,432.0,576.0,Grand Total,All staff,All staff
1,100654,1200,1,200,NaN,384.0,190.0,194.0,"Instructional, research and public service",All staff,"Instructional, research and public service staff"
2,100654,1210,1,210,NaN,379.0,188.0,191.0,Instructional staff,All staff,"Instructional staff, total"
3,100654,1220,1,220,NaN,5.0,2.0,3.0,Research,All staff,Research
4,100654,1250,1,250,NaN,25.0,4.0,21.0,"Librarians and Library Technicians/Archivists,...",All staff,Librarians/Library Technicians/Archivists and ...


## 10. Validate

Rules are declarative so the output is a persistable report: which checks ran, which failed, on how many rows, and which institutions were implicated. That report is the artefact you cite when claiming this table is fit for analysis.

In [10]:
RULES = [
    iu.unique_key('UNITID', 'STAFFCAT'),
    iu.in_range('HRTOTLT', 0, None),
]

report = iu.validate(curated, RULES, SLUG)
report.save(f"../reports/validation/{SLUG}.json")
display(report.to_frame()[["name", "status", "n_offending", "share", "note"]])

print("PASSED" if report.ok else "FAILED")
report.raise_if_failed()

,name,status,n_offending,share,note
0,"unique_key(UNITID,STAFFCAT)",pass,0,0.0,Declared grain must be unique
1,"in_range(HRTOTLT,0,None)",pass,0,0.0,Value plausibility bound


PASSED


Report(table='c11_hr', rows=180266, results=[{'name': 'unique_key(UNITID,STAFFCAT)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Declared grain must be unique', 'status': 'pass'}, {'name': 'in_range(HRTOTLT,0,None)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Value plausibility bound', 'status': 'pass'}], generated_utc='2026-09-24T17:19:37+00:00')

## 11. Write the curated table

The sidecar carries the reference period and grain with the data. This is the defence against assembling a panel by filename year when the underlying periods are offset differently per component.

In [11]:
path = iu.write_curated(
    curated,
    SLUG,
    root="../data/curated",
    reference_period=REFERENCE_PERIOD,
    grain=GRAIN,
    provenance=provenance,
    notes="S2023_NH is a flow, not a stock: it counts people hired between November 1, 2022 and October 31, 2023. Using it for staffing mix or tenure density measures the composition of one year's hiring rather than the composition of the workforce, and the two differ most at exactly the institutions undergoing change. Note also that STAFFCAT includes nested aggregate codes; summing across all of them double-counts.",
)

iu.write_provenance(provenance, f"../docs/provenance/{SLUG}.json")
print("wrote", path, f"({len(curated):,} rows x {curated.shape[1]} columns)")

wrote ../data/curated/c11_hr.parquet (180,266 rows x 11 columns)


## 12. Exercises

1. Re-run this notebook against the prior collection cycle by changing `TABLES`. The schema lock and the period assertion will both object; resolve each objection and record what changed between cycles.
2. Identify the three columns with the lowest reported-flag share, and argue whether each belongs in a predictive model at all.
3. Construct one derived cross-tabulation from this table, then apply `iu.suppress` and `iu.k_anonymity` to it. Report the smallest equivalence class before and after coarsening, and state the k you would require before publishing.
4. S2023_NH is a flow, not a stock: it counts people hired between November 1, 2022 and October 31, 2023. Write a validation rule that would catch this error if a colleague made it, and add it to `RULES` above.